In [ ]:
# =========================================================
# SINGLE-MODEL ROBUSTNESS EVALUATION: anymodel ex. Bilstm
# CSI-Bench Multitask/HumanActivityRecognition
# =========================================================

import os
import gc
import glob
import json
import h5py
import random
import shutil
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from IPython.display import FileLink, display

# =========================================================
# CHANGE THIS PATH IF NEEDED
# =========================================================

MODEL_NAME = "BiLSTM"
BILSTM_CKPT_PATH = "path of uploaded model"


OUT_DIR = "/kaggle/working/robustness_model"
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists(model_CKPT_PATH):
    raise FileNotFoundError(f"Checkpoint not found: {model_CKPT_PATH}")

# =========================================================
# CONFIG
# =========================================================

TASK_NAME = "HumanActivityRecognition"

SEED = 42
TARGET_SUBCARRIERS = 56
TARGET_TIME_LEN = 500

BATCH_SIZE = 64
NUM_WORKERS = 0
DROPOUT = 0.1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
    if hasattr(torch, "set_float32_matmul_precision"):
        torch.set_float32_matmul_precision("medium")

# =========================================================
# HELPERS
# =========================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def safe_torch_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def get_state_dict(ckpt):
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt:
        return ckpt["model_state_dict"]
    return ckpt

set_seed(SEED)
cleanup()

# =========================================================
# FIND CSI-BENCH ROOT
# =========================================================

def find_task_root(task_name="HumanActivityRecognition"):
    patterns = [
        f"/kaggle/input/datasets/guozhenjennzhu/csi-bench/Multitask/{task_name}",
        f"/kaggle/input/**/Multitask/{task_name}",
        f"/kaggle/input/**/{task_name}",
    ]

    candidates = []

    for pattern in patterns:
        for path in glob.glob(pattern, recursive=True):
            if os.path.isdir(path):
                candidates.append(path)

    candidates = list(dict.fromkeys(candidates))

    print("\nCandidate task roots:")
    for p in candidates:
        print(" -", p)

    for p in candidates:
        split_file = os.path.join(p, "splits", "test_id.json")
        metadata_file = os.path.join(p, "metadata", "sample_metadata.csv")

        if os.path.exists(split_file) and os.path.exists(metadata_file):
            print("\nUsing ROOT:")
            print(p)
            return p

    raise FileNotFoundError("Could not find CSI-Bench HumanActivityRecognition root.")

ROOT = find_task_root(TASK_NAME)

# =========================================================
# LOAD CHECKPOINT + LABEL MAPPING
# =========================================================

ckpt = safe_torch_load(model_CKPT_PATH, map_location="cpu")

if isinstance(ckpt, dict) and "label_mapping" in ckpt:
    LABEL_MAPPING = ckpt["label_mapping"]
else:
    metadata_path = os.path.join(ROOT, "metadata", "sample_metadata.csv")
    metadata_tmp = pd.read_csv(metadata_path)
    metadata_tmp["id"] = metadata_tmp["id"].astype(str)

    train_split_path = os.path.join(ROOT, "splits", "train_id.json")
    with open(train_split_path, "r") as f:
        train_ids_tmp = set(map(str, json.load(f)))

    train_meta_tmp = metadata_tmp[metadata_tmp["id"].isin(train_ids_tmp)]
    labels = sorted(train_meta_tmp["label"].unique())
    LABEL_MAPPING = {label: idx for idx, label in enumerate(labels)}

INV_LABEL_MAPPING = {v: k for k, v in LABEL_MAPPING.items()}
NUM_CLASSES = len(LABEL_MAPPING)

print("\nLabel mapping:")
print(LABEL_MAPPING)

# =========================================================
# DATASET
# =========================================================

class CSIBenchDataset(Dataset):
    def __init__(
        self,
        root_dir,
        split="test",
        normalize=True,
        target_subcarriers=56,
        target_time_len=500,
        label_mapping=None
    ):
        self.root_dir = root_dir
        self.split = split
        self.normalize = normalize
        self.target_subcarriers = target_subcarriers
        self.target_time_len = target_time_len
        self.label_mapping = label_mapping

        self.metadata_dir = os.path.join(root_dir, "metadata")
        self.splits_dir = os.path.join(root_dir, "splits")
        self.multitask_dir = os.path.dirname(root_dir)
        self.csi_bench_dir = os.path.dirname(self.multitask_dir)

        split_file = os.path.join(self.splits_dir, f"{split}_id.json")
        metadata_file = os.path.join(self.metadata_dir, "sample_metadata.csv")

        if not os.path.exists(split_file):
            raise FileNotFoundError(f"Missing split file: {split_file}")

        if not os.path.exists(metadata_file):
            raise FileNotFoundError(f"Missing metadata file: {metadata_file}")

        with open(split_file, "r") as f:
            self.sample_ids = list(map(str, json.load(f)))

        self.metadata = pd.read_csv(metadata_file)
        self.metadata["id"] = self.metadata["id"].astype(str)
        self.metadata["file_path"] = self.metadata["file_path"].astype(str)

        self.meta_dict = {
            str(row["id"]): row
            for _, row in self.metadata.iterrows()
        }

        print(f"Loaded {len(self.sample_ids)} samples for split: {split}")

    def __len__(self):
        return len(self.sample_ids)

    def resolve_file_path(self, raw_path):
        raw = str(raw_path).replace("\\", "/").strip()

        if raw.startswith("./"):
            raw = raw[2:]

        candidates = []

        if os.path.isabs(raw):
            candidates.append(os.path.normpath(raw))

        candidates.extend([
            os.path.normpath(os.path.join(self.metadata_dir, raw)),
            os.path.normpath(os.path.join(self.root_dir, raw)),
            os.path.normpath(os.path.join(self.multitask_dir, raw)),
            os.path.normpath(os.path.join(self.csi_bench_dir, raw)),
            os.path.normpath(os.path.join("/kaggle/input", raw)),
        ])

        if "sub_Human_h5/" in raw:
            suffix = raw.split("sub_Human_h5/", 1)[1]
            candidates.append(
                os.path.normpath(os.path.join(self.multitask_dir, "sub_Human_h5", suffix))
            )

        if "sub_Human_mat/" in raw:
            suffix = raw.split("sub_Human_mat/", 1)[1]
            candidates.append(
                os.path.normpath(os.path.join(self.multitask_dir, "sub_Human_mat", suffix))
            )

        for path in candidates:
            if os.path.exists(path):
                return path

        base = os.path.basename(raw)
        matches = []

        for search_base in [self.multitask_dir, self.csi_bench_dir]:
            pattern = os.path.join(search_base, "**", base)
            matches.extend(glob.glob(pattern, recursive=True))

        matches = sorted(list(set(matches)))

        if len(matches) == 1:
            return matches[0]

        if len(matches) > 1:
            raise RuntimeError(
                "Ambiguous file resolution. Multiple files share the same basename.\n"
                f"metadata file_path: {raw_path}\n"
                f"basename: {base}\n"
                "Matches:\n" + "\n".join(matches[:20])
            )

        raise FileNotFoundError(
            "Could not resolve CSI file path.\n"
            f"metadata file_path: {raw_path}\n"
            f"basename searched: {base}"
        )

    def load_h5(self, path):
        with h5py.File(path, "r") as f:
            keys = list(f.keys())

            for key in ["csi", "data", "CSI", "amplitude"]:
                if key in keys:
                    return f[key][:]

            return f[keys[0]][:]

    def to_ckt(self, data):
        data = np.array(data)

        if np.iscomplexobj(data):
            data = np.abs(data)

        data = data.astype(np.float32)

        if data.ndim == 2:
            a, b = data.shape

            if a <= b:
                return data[np.newaxis, :, :]
            else:
                return data.T[np.newaxis, :, :]

        if data.ndim == 3:
            s0, s1, s2 = data.shape

            if s0 <= 8 and s1 <= self.target_subcarriers * 2:
                return data

            if s2 <= 8 and s0 <= self.target_subcarriers * 2:
                return np.transpose(data, (2, 0, 1))

            if s2 <= 8 and s1 <= self.target_subcarriers * 2:
                return np.transpose(data, (2, 1, 0))

            if s2 <= 16:
                return np.transpose(data, (2, 0, 1))

        raise ValueError(f"Unexpected CSI shape: {data.shape}")

    def standardize_subcarriers(self, x):
        C, K, T = x.shape

        if K > self.target_subcarriers:
            x = x[:, :self.target_subcarriers, :]
        elif K < self.target_subcarriers:
            pad = np.zeros((C, self.target_subcarriers - K, T), dtype=x.dtype)
            x = np.concatenate([x, pad], axis=1)

        return x

    def standardize_time(self, x):
        C, K, T = x.shape

        if T > self.target_time_len:
            x = x[:, :, :self.target_time_len]
        elif T < self.target_time_len:
            pad = np.zeros((C, K, self.target_time_len - T), dtype=x.dtype)
            x = np.concatenate([x, pad], axis=2)

        return x

    def __getitem__(self, idx):
        sample_id = str(self.sample_ids[idx])

        if sample_id not in self.meta_dict:
            raise KeyError(f"Sample ID not found in metadata: {sample_id}")

        meta = self.meta_dict[sample_id]

        file_path = self.resolve_file_path(meta["file_path"])
        csi = self.load_h5(file_path)

        x = self.to_ckt(csi)
        x = self.standardize_subcarriers(x)
        x = self.standardize_time(x)

        if self.normalize:
            x = (x - x.mean()) / (x.std() + 1e-6)

        label_name = meta["label"]

        if label_name not in self.label_mapping:
            raise KeyError(f"Label not in mapping: {label_name}")

        y = self.label_mapping[label_name]

        return torch.from_numpy(x).float(), torch.tensor(y, dtype=torch.long)

test_dataset = CSIBenchDataset(
    ROOT,
    split="test",
    normalize=True,
    target_subcarriers=TARGET_SUBCARRIERS,
    target_time_len=TARGET_TIME_LEN,
    label_mapping=LABEL_MAPPING
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    drop_last=False,
    num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda"
)

sample_x, sample_y = test_dataset[0]
CSI_CHANNELS = sample_x.shape[0]

print("\n========== DATA INFO ==========")
print("Sample shape:", sample_x.shape)
print("CSI channels:", CSI_CHANNELS)
print("Num classes:", NUM_CLASSES)
print("Example label:", sample_y)

# =========================================================
# model MODEL DEFINITION
# Must match your baseline:
# hidden_dim=64, bidirectional=True, 2 layers
# Params should be around 187,061
# =========================================================

class Baselinemodel(nn.Module):
    def __init__(self):
        super().__init__()

        input_dim = CSI_CHANNELS * TARGET_SUBCARRIERS
        hidden_dim = 64

        self.input_proj = nn.Sequential(
            nn.LayerNorm(input_dim),
            nn.Linear(input_dim, hidden_dim),
            nn.GELU()
        )

        self.lstm = nn.LSTM(
            input_size=hidden_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=DROPOUT,
            bidirectional=True
        )

        self.head = nn.Sequential(
            nn.LayerNorm(hidden_dim * 2),
            nn.Linear(hidden_dim * 2, hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden_dim * 2, NUM_CLASSES)
        )

    def forward(self, x):
        B, C, K, T = x.shape
        x = x.reshape(B, C * K, T).transpose(1, 2)
        x = self.input_proj(x)
        out, _ = self.lstm(x)
        vec = out.mean(dim=1)
        return self.head(vec)

model = Baselinemodel().to(DEVICE)

params = sum(p.numel() for p in model.parameters())
print(f"\nmodel params from code: {params:,}")

state = get_state_dict(ckpt)
model.load_state_dict(state, strict=True)
model.eval()

print("\n✅ Loaded model checkpoint:")
print(model_CKPT_PATH)

# =========================================================
# CLEAN REPORT
# =========================================================

@torch.no_grad()
def evaluate_clean_report(model, loader):
    model.eval()

    preds_all = []
    labels_all = []

    for x, y in tqdm(loader, desc="Clean report", leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model(x)

        preds = logits.argmax(dim=1)

        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(y.detach().cpu().numpy())

    target_names = [INV_LABEL_MAPPING[i] for i in range(NUM_CLASSES)]

    report = classification_report(
        labels_all,
        preds_all,
        target_names=target_names,
        digits=4,
        zero_division=0
    )

    cm = confusion_matrix(labels_all, preds_all)

    print("\n========== CLEAN TEST REPORT ==========")
    print(report)

    print("\n========== CLEAN CONFUSION MATRIX ==========")
    print(cm)

    report_path = os.path.join(OUT_DIR, "model_clean_classification_report.txt")
    cm_path = os.path.join(OUT_DIR, "model_clean_confusion_matrix.csv")

    with open(report_path, "w") as f:
        f.write(report)

    pd.DataFrame(cm, index=target_names, columns=target_names).to_csv(cm_path)

    return labels_all, preds_all

clean_labels, clean_preds = evaluate_clean_report(model, test_loader)

# =========================================================
# PERTURBATIONS
# =========================================================

def perturb_clean(x):
    return x

def perturb_gaussian_noise(x, std=0.1):
    return x + torch.randn_like(x) * std

def perturb_random_subcarrier_mask(x, drop_prob=0.3):
    B, C, K, T = x.shape
    mask = (torch.rand(B, 1, K, 1, device=x.device) > drop_prob).float()
    return x * mask

def perturb_contiguous_subcarrier_mask(x, drop_ratio=0.3):
    B, C, K, T = x.shape
    width = max(1, int(K * drop_ratio))
    out = x.clone()

    for b in range(B):
        start = torch.randint(0, K - width + 1, (1,), device=x.device).item()
        out[b, :, start:start + width, :] = 0.0

    return out

def perturb_temporal_mask(x, drop_ratio=0.2):
    B, C, K, T = x.shape
    width = max(1, int(T * drop_ratio))
    out = x.clone()

    for b in range(B):
        start = torch.randint(0, T - width + 1, (1,), device=x.device).item()
        out[b, :, :, start:start + width] = 0.0

    return out

def perturb_time_shift(x, shift=50):
    return torch.roll(x, shifts=shift, dims=-1)

def perturb_crop_resize(x, keep_ratio=0.75):
    B, C, K, T = x.shape
    keep_len = max(8, int(T * keep_ratio))
    out_list = []

    for b in range(B):
        start = torch.randint(0, T - keep_len + 1, (1,), device=x.device).item()
        crop = x[b:b+1, :, :, start:start + keep_len]
        crop = crop.reshape(1, C * K, keep_len)

        resized = F.interpolate(
            crop,
            size=T,
            mode="linear",
            align_corners=False
        )

        resized = resized.reshape(1, C, K, T)
        out_list.append(resized)

    return torch.cat(out_list, dim=0)

def perturb_amplitude_scale(x, scale=0.7):
    return x * scale

def perturb_combined_mild(x):
    x = perturb_gaussian_noise(x, std=0.10)
    x = perturb_random_subcarrier_mask(x, drop_prob=0.20)
    x = perturb_temporal_mask(x, drop_ratio=0.10)
    return x

def perturb_combined_harsh(x):
    x = perturb_gaussian_noise(x, std=0.20)
    x = perturb_random_subcarrier_mask(x, drop_prob=0.40)
    x = perturb_temporal_mask(x, drop_ratio=0.20)
    x = perturb_crop_resize(x, keep_ratio=0.75)
    return x

ROBUSTNESS_CONDITIONS = [
    {"name": "clean", "fn": perturb_clean, "kwargs": {}, "trials": 1},

    {"name": "gaussian_noise_0.05", "fn": perturb_gaussian_noise, "kwargs": {"std": 0.05}, "trials": 3},
    {"name": "gaussian_noise_0.10", "fn": perturb_gaussian_noise, "kwargs": {"std": 0.10}, "trials": 3},
    {"name": "gaussian_noise_0.20", "fn": perturb_gaussian_noise, "kwargs": {"std": 0.20}, "trials": 3},

    {"name": "random_subcarrier_mask_10", "fn": perturb_random_subcarrier_mask, "kwargs": {"drop_prob": 0.10}, "trials": 3},
    {"name": "random_subcarrier_mask_30", "fn": perturb_random_subcarrier_mask, "kwargs": {"drop_prob": 0.30}, "trials": 3},
    {"name": "random_subcarrier_mask_50", "fn": perturb_random_subcarrier_mask, "kwargs": {"drop_prob": 0.50}, "trials": 3},

    {"name": "contiguous_subcarrier_mask_10", "fn": perturb_contiguous_subcarrier_mask, "kwargs": {"drop_ratio": 0.10}, "trials": 3},
    {"name": "contiguous_subcarrier_mask_30", "fn": perturb_contiguous_subcarrier_mask, "kwargs": {"drop_ratio": 0.30}, "trials": 3},

    {"name": "temporal_mask_10", "fn": perturb_temporal_mask, "kwargs": {"drop_ratio": 0.10}, "trials": 3},
    {"name": "temporal_mask_30", "fn": perturb_temporal_mask, "kwargs": {"drop_ratio": 0.30}, "trials": 3},

    {"name": "time_shift_25", "fn": perturb_time_shift, "kwargs": {"shift": 25}, "trials": 1},
    {"name": "time_shift_50", "fn": perturb_time_shift, "kwargs": {"shift": 50}, "trials": 1},

    {"name": "crop_resize_75", "fn": perturb_crop_resize, "kwargs": {"keep_ratio": 0.75}, "trials": 3},
    {"name": "crop_resize_50", "fn": perturb_crop_resize, "kwargs": {"keep_ratio": 0.50}, "trials": 3},

    {"name": "amplitude_scale_0.70", "fn": perturb_amplitude_scale, "kwargs": {"scale": 0.70}, "trials": 1},
    {"name": "amplitude_scale_1.30", "fn": perturb_amplitude_scale, "kwargs": {"scale": 1.30}, "trials": 1},

    {"name": "combined_mild", "fn": perturb_combined_mild, "kwargs": {}, "trials": 3},
    {"name": "combined_harsh", "fn": perturb_combined_harsh, "kwargs": {}, "trials": 3},
]

# =========================================================
# ROBUSTNESS EVALUATION
# =========================================================

@torch.no_grad()
def evaluate_model_under_condition(model, loader, condition, trial_seed=42):
    set_seed(trial_seed)

    model.eval()

    preds_all = []
    labels_all = []

    fn = condition["fn"]
    kwargs = condition["kwargs"]

    for x, y in tqdm(loader, desc=condition["name"], leave=False):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        x = fn(x, **kwargs)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model(x)

        preds = logits.argmax(dim=1)

        preds_all.extend(preds.detach().cpu().numpy())
        labels_all.extend(y.detach().cpu().numpy())

    return {
        "accuracy": float(accuracy_score(labels_all, preds_all)),
        "macro_f1": float(f1_score(labels_all, preds_all, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(labels_all, preds_all, average="weighted", zero_division=0)),
    }

def evaluate_trials(model, model_name, loader, condition):
    trial_results = []

    for t in range(condition["trials"]):
        trial_seed = SEED + 1000 * t

        result = evaluate_model_under_condition(
            model=model,
            loader=loader,
            condition=condition,
            trial_seed=trial_seed
        )

        trial_results.append(result)

    accs = np.array([r["accuracy"] for r in trial_results])
    macros = np.array([r["macro_f1"] for r in trial_results])
    weighteds = np.array([r["weighted_f1"] for r in trial_results])

    return {
        "model": model_name,
        "condition": condition["name"],
        "trials": condition["trials"],

        "acc_mean": float(accs.mean()),
        "acc_std": float(accs.std(ddof=1)) if len(accs) > 1 else 0.0,

        "macro_f1_mean": float(macros.mean()),
        "macro_f1_std": float(macros.std(ddof=1)) if len(macros) > 1 else 0.0,

        "weighted_f1_mean": float(weighteds.mean()),
        "weighted_f1_std": float(weighteds.std(ddof=1)) if len(weighteds) > 1 else 0.0,
    }

all_rows = []

for condition in ROBUSTNESS_CONDITIONS:
    print("\n" + "=" * 80)
    print("Condition:", condition["name"])
    print("=" * 80)

    cleanup()

    row = evaluate_trials(
        model=model,
        model_name=MODEL_NAME,
        loader=test_loader,
        condition=condition
    )

    all_rows.append(row)

    print(
        f"{MODEL_NAME:22s} | "
        f"Acc: {row['acc_mean']*100:.2f} ± {row['acc_std']*100:.2f} | "
        f"Macro-F1: {row['macro_f1_mean']*100:.2f} ± {row['macro_f1_std']*100:.2f} | "
        f"Weighted-F1: {row['weighted_f1_mean']*100:.2f} ± {row['weighted_f1_std']*100:.2f}"
    )

results_df = pd.DataFrame(all_rows)

# =========================================================
# DROPS FROM CLEAN
# =========================================================

clean_row = results_df[results_df["condition"] == "clean"].iloc[0]

clean_acc = clean_row["acc_mean"]
clean_macro_f1 = clean_row["macro_f1_mean"]
clean_weighted_f1 = clean_row["weighted_f1_mean"]

drop_rows = []

for _, row in results_df.iterrows():
    out = row.to_dict()

    out["acc_drop"] = clean_acc - row["acc_mean"]
    out["macro_f1_drop"] = clean_macro_f1 - row["macro_f1_mean"]
    out["weighted_f1_drop"] = clean_weighted_f1 - row["weighted_f1_mean"]

    drop_rows.append(out)

results_drop_df = pd.DataFrame(drop_rows)

display_df = results_drop_df.copy()

percent_cols = [
    "acc_mean", "acc_std",
    "macro_f1_mean", "macro_f1_std",
    "weighted_f1_mean", "weighted_f1_std",
    "acc_drop", "macro_f1_drop", "weighted_f1_drop",
]

for c in percent_cols:
    display_df[c] = display_df[c] * 100

# =========================================================
# SAVE RESULTS
# =========================================================

raw_path = os.path.join(OUT_DIR, "robustness_raw_model.csv")
drop_path = os.path.join(OUT_DIR, "robustness_with_drops_percent_model.csv")

results_df.to_csv(raw_path, index=False)
display_df.to_csv(drop_path, index=False)

print("\nSaved:")
print(raw_path)
print(drop_path)

print("\n========== model ROBUSTNESS RESULTS WITH DROPS (%) ==========")
display(display_df)

# =========================================================
# ZIP RESULTS
# =========================================================

zip_path = "/kaggle/working/robustness_model.zip"

if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive(
    base_name=zip_path.replace(".zip", ""),
    format="zip",
    root_dir=OUT_DIR
)

print("\nDownload results:")
display(FileLink(zip_path))

cleanup()